# Workit 법령 임베딩 — 4개 데이터셋 (jo / jo_fixedid / ho / ho_fixedid)

- 모델: **BGE-M3** (`BAAI/bge-m3`) 단일 — dense+sparse hybrid 지원 때문에 이미 확정된 선택
- 평가지표: Recall@1 / Recall@5 / Recall@10 / MRR (NDCG@10은 정답 1개짜리 synthetic eval에선
  MRR과 사실상 같은 정보라 여기선 생략 — RAG 파이프라인 평가 쪽에서 계속 사용)
- 입력: `merge_chunks.py`로 만든 4개 flat chunk 리스트 파일
  (`chunks_jo.json`, `chunks_jo_fixedid.json`, `chunks_ho.json`, `chunks_ho_fixedid.json`)


In [1]:
# ── 셀 1: GPU 확인 ──────────────────────────────────────────
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


CUDA 사용 가능: True
GPU: Tesla T4


In [2]:
# ── 셀 2: 패키지 설치 ────────────────────────────────────────
!pip install -q sentence-transformers FlagEmbedding


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.0 MB/s eta 0:00:00


In [10]:
# ── 셀 3: chunks 4개 파일 업로드 ─────────────────────────────
# merge_chunks.py 출력물을 그대로 업로드:
#   chunks_jo.json, chunks_jo_fixedid.json, chunks_ho.json, chunks_ho_fixedid.json
import os

DATASET_FILES = {
    'jo':          'chunks_jo.json',
    'jo_fixedid':  'chunks_jo_fixedid.json',
    'ho':          'chunks_ho.json',
    'ho_fixedid':  'chunks_ho_fixedid.json',
}

for fname in DATASET_FILES.values():
    if os.path.exists(fname):
        os.remove(fname)
        print(f'기존 파일 삭제: {fname}')

from google.colab import files
uploaded = files.upload()  # 4개 파일 한 번에 선택


기존 파일 삭제: chunks_jo.json
기존 파일 삭제: chunks_jo_fixedid.json
기존 파일 삭제: chunks_ho.json
기존 파일 삭제: chunks_ho_fixedid.json


Saving chunks_ho.json to chunks_ho.json
Saving chunks_ho_fixedid.json to chunks_ho_fixedid.json
Saving chunks_jo.json to chunks_jo.json
Saving chunks_jo_fixedid.json to chunks_jo_fixedid.json


In [11]:
# ── 셀 4: 4개 데이터셋 로드 ───────────────────────────────────
# merge_chunks.py에서 이미 chunk_id 기준 dedupe를 했지만, 안전하게 한 번 더 확인만 함.
# (jo/ho가 이미 폴더/파일로 분리되어 있어서 예전 노트북의 is_parent 필터링은 더 이상 필요 없음)
import json

datasets = {}  # name -> {'texts': [...], 'chunk_ids': [...], 'chunks': [...]}

for name, fname in DATASET_FILES.items():
    with open(fname, encoding='utf-8') as f:
        chunks = json.load(f)

    seen = set()
    deduped = []
    for c in chunks:
        cid = c['chunk_id']
        if cid in seen:
            continue
        seen.add(cid)
        deduped.append(c)

    datasets[name] = {
        'chunks':    deduped,
        'texts':     [c['text'] for c in deduped],
        'chunk_ids': [c['chunk_id'] for c in deduped],
    }
    print(f"{name:12s} {len(chunks):>6,}개 -> dedupe 후 {len(deduped):>6,}개")


jo            1,126개 -> dedupe 후  1,126개
jo_fixedid    1,126개 -> dedupe 후  1,126개
ho            7,640개 -> dedupe 후  7,640개
ho_fixedid    6,890개 -> dedupe 후  6,890개


In [12]:
# ── 셀 5: 4개 데이터셋 각각 평가셋 생성 ────────────────────────
import random
random.seed(42)  # 재현 가능하게 시드 고정

def make_eval_dataset(chunks, n=200):
    templates = [
        '다음 내용은 어떤 규정을 설명하나요?',
        '이 조항은 무엇에 관한 내용인가요?',
        '다음 규정과 관련된 조문을 찾아주세요.',
        '계약 실무에서 아래 내용은 어떤 법적 근거에 해당하나요?'
    ]
    eval_data = []
    qid = 1
    for chunk in chunks:
        text = chunk['text']
        if len(text) < 80:
            continue
        query = random.choice(templates) + '\n\n' + text[:120]
        eval_data.append({
            'query_id':      f'eval_{qid:03d}',
            'query':         query,
            'relevant_docs': [chunk['chunk_id']]
        })
        qid += 1
    random.shuffle(eval_data)
    return eval_data[:n]

for name in datasets:
    eval_data = make_eval_dataset(datasets[name]['chunks'], n=200)
    datasets[name]['eval_data'] = eval_data
    with open(f'eval_dataset_{name}.json', 'w', encoding='utf-8') as f:
        json.dump(eval_data, f, ensure_ascii=False, indent=2)
    print(f"{name:12s} 평가셋 {len(eval_data)}개")


jo           평가셋 200개
jo_fixedid   평가셋 200개
ho           평가셋 200개
ho_fixedid   평가셋 200개


In [13]:
# ── 셀 6: 모델 평가 함수 (Recall@1/5/10, MRR) ──────────────────
import gc
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_model(model_name, texts, chunk_ids, eval_data, batch_size=8):
    """
    인자:
        texts      — 임베딩할 청크 텍스트 리스트
        chunk_ids  — 청크 ID 리스트 (texts와 순서 동일)
        eval_data  — 평가셋 (query_id / query / relevant_docs, 정답 1개씩)
    """
    model = SentenceTransformer(model_name, device='cuda')

    doc_vectors = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    recall1 = recall5 = recall10 = mrr = 0

    for item in eval_data:
        gt_docs = set(item['relevant_docs'])
        qvec = model.encode(
            item['query'],
            normalize_embeddings=True,
            convert_to_numpy=True
        )
        scores     = cosine_similarity([qvec], doc_vectors)[0]
        ranked_ids = [chunk_ids[i] for i in np.argsort(scores)[::-1]]

        if any(d in gt_docs for d in ranked_ids[:1]):  recall1  += 1
        if any(d in gt_docs for d in ranked_ids[:5]):  recall5  += 1
        if any(d in gt_docs for d in ranked_ids[:10]): recall10 += 1
        for rank, d in enumerate(ranked_ids, 1):
            if d in gt_docs:
                mrr += 1 / rank
                break

    n = len(eval_data)
    del doc_vectors, model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        'Recall@1':  round(recall1  / n, 4),
        'Recall@5':  round(recall5  / n, 4),
        'Recall@10': round(recall10 / n, 4),
        'MRR':       round(mrr      / n, 4),
    }


In [14]:
# ── 셀 7: BGE-M3로 4개 데이터셋 전부 평가 ─────────────────────
import pandas as pd

results = []
for name in ['jo', 'jo_fixedid', 'ho', 'ho_fixedid']:
    print(f"\n▶ {name} 평가 중...")
    d = datasets[name]
    r = evaluate_model('BAAI/bge-m3', d['texts'], d['chunk_ids'], d['eval_data'], batch_size=8)
    r['Dataset'] = name
    results.append(r)

result_df = pd.DataFrame(results)[['Dataset', 'Recall@1', 'Recall@5', 'Recall@10', 'MRR']]
display(result_df)

result_df.to_csv('embedding_eval_results.csv', index=False)
print('\n저장 완료: embedding_eval_results.csv')



▶ jo 평가 중...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/141 [00:00<?, ?it/s]


▶ jo_fixedid 평가 중...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

...


▶ ho_fixedid 평가 중...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/862 [00:00<?, ?it/s]

,Dataset,Recall@1,Recall@5,Recall@10,MRR
0,jo,0.990,1.000,1.0,0.9942
1,jo_fixedid,0.980,0.995,1.0,0.9859
2,ho,0.990,1.000,1.0,0.9950
3,ho_fixedid,0.995,1.000,1.0,0.9975



저장 완료: embedding_eval_results.csv


In [16]:
# ── 셀 8: BGE-M3 dense + sparse 임베딩 및 저장 (4개 데이터셋 전부) ──
from FlagEmbedding import BGEM3FlagModel
import numpy as np
import json

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

saved_files = []

for name in ['jo', 'jo_fixedid', 'ho', 'ho_fixedid']:
    d = datasets[name]
    print(f"\n=== {name} 임베딩 ({len(d['texts'])}개) ===")

    output = model.encode(
        d['texts'],
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
        batch_size=8,
    )

    vec_path = f'vectors_{name}.npz'
    np.savez(
        vec_path,
        vectors=output['dense_vecs'],
        chunk_ids=np.array(d['chunk_ids'])
    )

    sparse_path = f'sparse_weights_{name}.json'
    sparse_weights = [{k: float(v) for k, v in sw.items()} for sw in output['lexical_weights']]
    with open(sparse_path, 'w', encoding='utf-8') as f:
        json.dump(sparse_weights, f)

    print(f"  dense shape: {output['dense_vecs'].shape}")
    print(f"  -> saved: {vec_path}, {sparse_path}")
    saved_files.extend([vec_path, sparse_path])

del model
gc.collect()
torch.cuda.empty_cache()

print('\n✅ 4개 데이터셋 임베딩 전부 완료:', saved_files)


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


=== jo 임베딩 (1126개) ===


Inference Embeddings: 100%|██████████| 141/141 [00:10<00:00, 13.80it/s]


  dense shape: (1126, 1024)
  -> saved: vectors_jo.npz, sparse_weights_jo.json

=== jo_fixedid 임베딩 (1126개) ===


...

  dense shape: (1126, 1024)
  -> saved: vectors_jo_fixedid.npz, sparse_weights_jo_fixedid.json

=== ho 임베딩 (7640개) ===


Inference Embeddings: 100%|██████████| 955/955 [00:19<00:00, 49.18it/s]


  dense shape: (7640, 1024)
  -> saved: vectors_ho.npz, sparse_weights_ho.json

=== ho_fixedid 임베딩 (6890개) ===


Inference Embeddings: 100%|██████████| 862/862 [00:18<00:00, 45.71it/s]


  dense shape: (6890, 1024)
  -> saved: vectors_ho_fixedid.npz, sparse_weights_ho_fixedid.json

✅ 4개 데이터셋 임베딩 전부 완료: ['vectors_jo.npz', 'sparse_weights_jo.json', 'vectors_jo_fixedid.npz', 'sparse_weights_jo_fixedid.json', 'vectors_ho.npz', 'sparse_weights_ho.json', 'vectors_ho_fixedid.npz', 'sparse_weights_ho_fixedid.json']


In [17]:
# ── 셀 9: 전체 다운로드 ──────────────────────────────────────
from google.colab import files as colab_files

for f in saved_files:
    colab_files.download(f)
colab_files.download('embedding_eval_results.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>